<a href="https://colab.research.google.com/github/jimhopgtu/google-ai-agents-daily-content-tool/blob/main/Daily_Relevant_Content.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.adk.agents import Agent, SequentialAgent, ParallelAgent, LoopAgent
from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner
from google.adk.tools.google_search_tool import GoogleSearchTool
from google.adk.tools import FunctionTool
from google.genai import types
import json
from pydantic import BaseModel, Field
from typing import List
import os
from google.colab import userdata, drive
from datetime import datetime
import pytz
eastern_tz = pytz.timezone('America/New_York')

print("✅ ADK components imported successfully.")

✅ ADK components imported successfully.


In [2]:

# This pulls the secret you just created and sets it as an environment variable
# Most Google SDKs (including ADK) look for "GOOGLE_API_KEY" automatically.
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

print("✅ API Key successfully loaded into environment!")

# from IPython.core.display import display, HTML
retry_config=types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier
    initial_delay=1, # Initial delay before first retry (in seconds)
    http_status_codes=[429, 500, 503, 504] # Retry on these HTTP errors
)


# Ensure Drive is mounted correctly
drive.mount('/content/drive', force_remount=True)

print("Executed at:", datetime.now(eastern_tz))

✅ API Key successfully loaded into environment!
Executed at: 2025-12-19 06:35:19.356800-05:00


In [3]:
search_instruction = """
You are a Research Scout for an Analytics Leader.
Your goal is to provide a balanced mix of content . For every run, you MUST use the search tool to find:

1. THE LATEST (24h): Top 3 industry-shifting news (e.g., Anthropic, OpenAI, Google).
2. THE ARCHITECTURE: Top 1 technical blog posts from engineering-heavy companies
   (e.g., MongoDB, Pinecone, Meta Engineering) that discuss 'why' or 'how'—not just 'what'.
3. THE STACK: Top 2 recent updates from the modern BI stack (e.g., dbt, Snowflake, Databricks, BigQuery, Looker, Atlan, PowerBI, Tableau)
4. THE LOCAL: 1 AI event in the NYC area or a major remote global summit.
5. Top 1 Obsidian plug in or use case that are new and could be useful.

## CRITICAL CONSTRAINTS
<Constraints>
- ANTI-FLUFF: If results are 'marketing fluff', refine keywords to include 'technical deep dive'.
- OUTPUT FORMAT: Return ONLY the URL and a 1-2 sentence summary. No full articles.
- MULTIMEDIA: YouTube videos are acceptable.
</Constraints>

## OUTPUT FORMAT (MANDATORY)
For every single item you find, you MUST follow this exact format:
- **Title**: [Name of the article/video]
- **Source URL**: [Insert the full direct link here]
- **Summary**: [1-2 sentences of why this matters for a data leader]

## CRITICAL RULES
- NEVER provide a news item without a corresponding URL.
- If you find a great story but the URL is missing from your tool output, do not include the story.
"""

# Define the data structure as a Python list/dictionary
LEADER_CONTEXT_DATA = {
    "ANALYTICS_LEADER_CONTEXT": [
        {
            "Category": "Modeling",
            "Shift": "Causal Inference (MMM/MTA), Advanced Models",
            "Action": "Prioritize Experimentation and Causal Strategy (A/B testing, incrementality)"
        },
        {
            "Category": "Architecture",
            "Shift": "Semantic Layer is SOT, often led by **Knowledge Engineer**",
            "Action": "Architect **AI Trust** and robust **Data Governance**"
        },
        {
            "Category": "Role & Skills",
            "Shift": "Analyst as Prompt Engineer/Consultant",
            "Action": "Coach for **Business Acumen** (the 'Why') and focus on recommendations"
        },
        {
            "Category": "Specialization",
            "Shift": "Deep expertise in one major stack (e.g., GCP, Azure)",
            "Action": "Standardize and Optimize the chosen stack to maximize value"
        },
        {
            "Category": "Analyst Profile",
            "Shift": "**Hybrid Role** (Business SME + Data Engineering + **Knowledge Engineering**)",
            "Action": "Redefine career path, mandate **DE fundamentals** and context structuring"
        },
        {
            "Category": "Governance",
            "Shift": "Mandatory focus on **Data Governance** and **AI TRiSM**",
            "Action": "Audit AI outputs, enforce lineage, and ensure ethical compliance"
        },
        {
            "Category": "Speed",
            "Shift": "Shift to **Real-Time** and **Edge Analytics**",
            "Action": "Invest in Modern Architecture (e.g., Data Mesh) for streaming data processing"
        },
        {
            "Category": "Leadership",
            "Shift": "Highest value in Human-Centric/Soft Skills and organizational influence",
            "Action": "Cultivate critical thinking, emotional intelligence, and **cross-department bridge-building** to remove data silos"
        }
    ]
}

# Convert it to a pretty-printed string ONLY when you need to feed it to the Agent
context_string = json.dumps(LEADER_CONTEXT_DATA, indent=2)



relevance_instruction = f"""
## ROLE
You are a Quality Controller for an Analytics Leader.

## DATA SOURCES
1. **NEW ARTICLES**: These are the latest findings to evaluate:
   {{raw_news_data}}

2. **CURRENT VAULT**: These are articles you already approved in previous rounds:
   {{approved_stories?}}

## STRATEGIC CONTEXT
Compare the NEW ARTICLES against these priorities:
{context_string}

## YOUR TASK
1. **Filter**: Read every article in 'NEW ARTICLES'.
2. **Score**: Assign a score (1-5).
3. **Selection**: If a new article has a score >= 3, add it to the list of 'approved_stories'.
4. **Goal Check**:
   - If the total count of approved stories (New Approved + Current Vault) is 8 or more, you MUST call the `exit_loop` function and nothing else.
   - Otherwise, find new articles for approval.

## OUTPUT
Return a structured JSON matching the RelevanceResponse schema. Include only articles with Score >= 3.

- You MUST return RAW JSON only.
- DO NOT wrap the output in markdown code blocks (e.g., no ```json).
- DO NOT include any preamble like "Here is the JSON:".
- The output must start with {{ and end with }}.
"""


class ScoredArticle(BaseModel):
    title: str
    url: str
    category: str = Field(description="The category from the Analytics Leader Context")
    score: int = Field(description="Relevance score from 1-5")
    action: str = Field(description="Actionable advice for the leader")
    summary: str

class RelevanceResponse(BaseModel):
    evaluated_articles: List[ScoredArticle]

print("Executed at:", datetime.now(eastern_tz))

Executed at: 2025-12-19 06:35:19.382575-05:00


In [4]:
# This is the function that the RefinerAgent will call to exit the loop.
def exit_loop():
    """Call this function ONLY when the critique is 'APPROVED', indicating the story is finished and no more changes are needed."""
    return {"status": "approved", "message": "Story approved. Exiting refinement loop."}


print("✅ exit_loop function created.")
print("Executed at:", datetime.now(eastern_tz))

✅ exit_loop function created.
Executed at: 2025-12-19 06:35:19.391858-05:00


In [5]:

# 1. Search Agent
search_agent = Agent(
    name="news_finder",
    model="gemini-2.0-flash", # Use 2.0 for speed/tool use gemini-2.0-flash gemini-1.5-flash or gemini-1.5-flash-8b
    tools=[GoogleSearchTool()],
    instruction=search_instruction,
    output_key="raw_news_data"
)

# 2. Relevance Agent
# Update the agent to save its "Approved" list to the vault
relevance_agent = Agent(
    name="relevance_agent",
    model="gemini-2.0-flash",
    output_schema=RelevanceResponse,
    instruction=relevance_instruction,
    output_key="approved_stories" # This overwrites/updates the vault
    ,
tools=[FunctionTool(exit_loop)]
)

# 3. Loop Agent
story_refinement_loop = LoopAgent(
    name="StoryRefinementLoop",
    sub_agents=[search_agent, relevance_agent],
    # The SDK usually looks for a single condition string referencing the state
    max_iterations=3
)

# 4. Reporting Agent
reporting_agent = Agent(
    name="reporting_agent",
    model="gemini-2.0-flash",
    instruction="""
    ## TASK
    1. Read the articles stored in {{approved_stories?}}.
    2. If the vault is empty, simply state "No highly relevant news found for today."
    3. If articles exist, format them into a professional Markdown report.

    ## FORMAT REQUIREMENTS
    - Use ## Headers for different categories.
    - Use **Bolding** for key takeaways.
    - Ensure every Title is a clickable [Markdown Link](URL).
    - Provide a "Why this matters" section for each article.
    """
)

print("Executed at:", datetime.now(eastern_tz))

Executed at: 2025-12-19 06:35:19.408514-05:00


In [6]:
# 1. Start with the Search Agent
runner = InMemoryRunner(agent=search_agent)

# 2. RUN SEARCH (This creates raw_news_data)
print("🔍 Searching...")
await runner.run_debug(
    "Find news for the analytics leader.",
    session_id="daily_report_session"
)

# 3. SWAP THE AGENT MANUALLY (The 'Brain' Swap)
# This keeps the session alive but changes the instructions
runner.agent = relevance_agent

# 4. RUN RELEVANCE (This reads raw_news_data)
print("⚖️ Evaluating...")
try:
    await runner.run_debug(
        "Evaluate the data in raw_news_data.",
        session_id="daily_report_session"
    )
except Exception as e:
    # If the backtick error happens here, we can catch it
    print(f"Validation Error: {e}")

# 5. SWAP TO REPORTER
runner.agent = reporting_agent

# 6. RUN REPORT
print("📊 Reporting...")
final_report = await runner.run_debug(
    "Generate the markdown report from approved_stories.",
    session_id="daily_report_session"
)

# run_debug returns a list of events. We want the text from the last event.
print(final_report[-1].content)
print("Executed at:", datetime.now(eastern_tz))

🔍 Searching...

 ### Created new session: daily_report_session

User > Find news for the analytics leader.


news_finder > Okay, I will find the news for the analytics leader, following the specified format and constraints.

Here's the news for the analytics leader:

- **Title**: Microsoft, Google, OpenAI, and Anthropic join forces to form Agentic AI alliance, according to report — organization backed by the Linux Foundation is set to create open source standards for AI agents
- **Source URL**: https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFplKVqHVoazqK53n-b6IZo52yWq2SrIzUN-VsSpYCTuD84H7-XTOb5S1I9SAmYkpO0iaTEpL305AxdYmQrUIm7q5GCkJ2y_wqLvNBnbkruwE6TmP_XvDetkAdv3jeNs9hj1_4mW_dxQmO994plaXLZJw0qhEJl7dAJLyhiElAhBwVYImYlIFAH9TiNQlS46YIP2ICo318-4whcpb1jU1yihDgQznGvILHgPb--SqBEpxcBHt08a9GtKFSLi1Jp64SnhFxLFnggEMJbevuav39OoVrfEKzw9Sxf6lez5GvByzPPfI7qjdEK64BgsvFyAxEhsyxFjhFVc5a66iVqIY2I-kU14R6kwgVm5WixUqvaqK0u4GEksc08ymHnMa_Wtzg0OnjsrXJ-WfUO1cmpUntOlqw2TESunLwCBOZ7toBbIPY=
- **Summary**: Microsoft, Google, OpenAI, and Anthropic are collaborating to establish the Agentic AI Foundat

Validation Error: 1 validation error for RelevanceResponse
  Invalid JSON: expected value at line 1 column 1 [type=json_invalid, input_value='```json\n{\n  "evaluated...: 4\n    }\n  ]\n}\n```', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/json_invalid
📊 Reporting...

 ### Continue session: daily_report_session

User > Generate the markdown report from approved_stories.
reporting_agent > ## AI & Analytics News for Today

### AI Development & Strategy

- **[Microsoft, Google, OpenAI, and Anthropic join forces to form Agentic AI alliance, according to report](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFplKVqHVoazqK53n-b6IZo52yWq2SrIzUN-VsSpYCTuD84H7-XTOb5S1I9SAmYkpO0iaTEpL305AxdYmQrUIm7q5GCkJ2y_wqLvNBnbkruwE6TmP_XvDetkAdv3jeNs9hj1_4mW_dxQmO994plaXLZJw0qhEJl7dAJLyhiElAhBwVYImYlIFAH9TiNQlS46YIP2ICo318-4whcpb1jU1yihDgQznGvILHgPb--SqBEpxcBHt08a9GtKFSLi1Jp64SnhFxLFnggEMJbevuav39OoVrfEKzw9Sxf6lez5GvByzPPfI7qjdEK64BgsvFyAxEhsyxFj

In [8]:


# 1. Setup the Path - Using 'MyDrive' (no space) is often more reliable for syncing
# We add a timestamp (hour/minute) to ensure this is a NEW file, bypassing any 'deleted file' cache
today_date = datetime.now().strftime("%Y-%m-%d_%H%M")
filename = f"{today_date}_Analytics_Report.md"
save_path = "/content/drive/MyDrive/AI/Obsidian Vault/daily news/"

# 2. Create folder if it's missing
if not os.path.exists(save_path):
    print(f"Creating missing directory: {save_path}")
    os.makedirs(save_path, exist_ok=True)

full_path = os.path.join(save_path, filename)

# 3. Extract and Clean the Report
try:
    # Look back for the actual text content
    final_report_text = ""
    for event in reversed(final_report):
        if event.content and event.content.parts:
            text_parts = [p.text for p in event.content.parts if p.text]
            if text_parts:
                final_report_text = "\n".join(text_parts)
                break

    # Remove the ```markdown code block wrappers
    clean_report = re.sub(r'^```(?:markdown)?\n?|```$', '', final_report_text.strip(), flags=re.MULTILINE)

    # 4. Write the file and FORCE a sync
    with open(full_path, "w") as f:
        f.write(clean_report)
        f.flush() # Force write to the OS buffer
        os.fsync(f.fileno()) # Force write to the disk

    print(f"✅ Success! File physically written.")
    print(f"📂 New Filename: {filename}")
    print(f"📍 Full Path: {full_path}")

except Exception as e:
    print(f"❌ Error during saving: {e}")

# 5. Final verification - this 'pings' the file so Drive sees it
!ls -l "{full_path}"

Mounted at /content/drive
✅ Success! File physically written.
📂 New Filename: 2025-12-19_1200_Analytics_Report.md
📍 Full Path: /content/drive/MyDrive/AI/Obsidian Vault/daily news/2025-12-19_1200_Analytics_Report.md
-rw------- 1 root root 5916 Dec 19 12:00 '/content/drive/MyDrive/AI/Obsidian Vault/daily news/2025-12-19_1200_Analytics_Report.md'


In [82]:
# !rm -rf /content/drive